# 文件查看与编辑

上一节学习了文件和目录操作。本节关注如何在命令行中查看文本文件，并用简单命令完成轻量编辑。

## 学习目标

完成本节后，你应该能够：

- 用 `cat` 快速输出短文件内容。
- 用 `less` 分页查看较长文件。
- 用 `head` 和 `tail` 查看文件开头和结尾。
- 用 `sed` 做简单的文本替换或按行查看。
- 判断哪些命令只查看文件，哪些命令会修改输出或文件。

## 前提准备

先通过 `cd` 命令进入本章目录。下面的命令假设你已经在课程项目根目录。

```bash
cd books/01_Shell入门
pwd
ls -la
```

确认当前目录中能看到 `notes.txt` 和 `resnet.py`。本节用 `notes.txt` 练习短文本查看，用 `resnet.py` 练习较长代码文件查看。

## `find`：查找文件

`find` 用来在目录中查找文件。常见格式是 `find 查找位置 -name "文件名模式"`。

下面的命令表示：从当前目录 `./` 开始，查找所有以 `.py` 结尾的文件。

In [29]:
%%bash
find ./ -name "*.py"

./resnet.py


## `cat` 与 `less`

`cat` 适合查看短文件，输出会直接打印到终端：



In [5]:
cat notes.txt

This is a simple note.



`less` 适合查看较长文件，可以上下翻页：



```bash
less resnet.py
```



在 `less` 中常用按键：

| 按键 | 作用 |
| --- | --- |
| `Space` | 向下翻页 |
| `b` | 向上翻页 |
| `/关键词` | 搜索关键词 |
| `q` | 退出 |

## `head` 与 `tail`

`head` 查看文件开头，`tail` 查看文件结尾：



In [15]:
%%bash
head resnet.py

from functools import partial
from typing import Any, Callable, Optional, Union

import torch
import torch.nn as nn
from torch import Tensor

from ..transforms._presets import ImageClassification
from ..utils import _log_api_usage_once
from ._api import register_model, Weights, WeightsEnum


In [16]:
%%bash
tail resnet.py

            base class. Please refer to the `source code
            <https://github.com/pytorch/vision/blob/main/torchvision/models/resnet.py>`_
            for more details about this class.
    .. autoclass:: torchvision.models.Wide_ResNet101_2_Weights
        :members:
    """
    weights = Wide_ResNet101_2_Weights.verify(weights)

    _ovewrite_named_param(kwargs, "width_per_group", 64 * 2)
    return _resnet(Bottleneck, [3, 4, 23, 3], weights, progress, **kwargs)




也可以指定行数：



In [18]:
%%bash
head -n 5 notes.txt


This is a simple note.

In [17]:
%%bash
tail -n 5 notes.txt

This is a simple note.



查看日志时，`tail` 很常用。下面的命令会持续显示文件末尾的新内容：



In [ ]:
tail -f app.log



按 `Ctrl+C` 可以停止持续查看。

## `grep`：搜索文本
`grep` 用来在文件中搜索包含指定文本的行，基本格式是 `grep 要搜索的文本 文件名`。

In [28]:
%%bash
grep num_classes resnet.py

        num_classes: int = 1000,
        self.fc = nn.Linear(512 * block.expansion, num_classes)
        _ovewrite_named_param(kwargs, "num_classes", len(weights.meta["categories"]))


## `sed`：简单文本处理

`sed` 来自 stream editor，常用于按行处理文本。本节只掌握三个常见用法：按行号查看、按模式匹配、替换输出。

先读懂后面会用到的几个符号：

| 写法 | 含义 |
| --- | --- |
| `-n` | 关闭默认输出，只打印明确要求打印的内容 |
| `p` | print，打印匹配到的行 |
| `1,5p` | 打印第 1 到第 5 行 |
| `/模式/p` | 打印匹配“模式”的行，`/` 用来包住要匹配的文本或正则表达式 |
| `s/旧文本/新文本/` | substitute，表示把旧文本替换成新文本 |

因此，`sed -n '1,5p' resnet.py` 的意思是：不默认输出，只打印 `resnet.py` 的第 1 到第 5 行。

In [21]:
%%bash
# 查看第 1 到第 5 行
sed -n '1,5p' resnet.py

from functools import partial
from typing import Any, Callable, Optional, Union

import torch
import torch.nn as nn


也可以按模式查看指定内容。例如，下面的命令会打印所有以 `def ` 开头的函数定义行。

In [22]:
%%bash
sed -n '/^def /p' resnet.py

def conv3x3(in_planes: int, out_planes: int, stride: int = 1, groups: int = 1, dilation: int = 1) -> nn.Conv2d:
def conv1x1(in_planes: int, out_planes: int, stride: int = 1) -> nn.Conv2d:
def _resnet(
def resnet18(*, weights: Optional[ResNet18_Weights] = None, progress: bool = True, **kwargs: Any) -> ResNet:
def resnet34(*, weights: Optional[ResNet34_Weights] = None, progress: bool = True, **kwargs: Any) -> ResNet:
def resnet50(*, weights: Optional[ResNet50_Weights] = None, progress: bool = True, **kwargs: Any) -> ResNet:
def resnet101(*, weights: Optional[ResNet101_Weights] = None, progress: bool = True, **kwargs: Any) -> ResNet:
def resnet152(*, weights: Optional[ResNet152_Weights] = None, progress: bool = True, **kwargs: Any) -> ResNet:
def resnext50_32x4d(
def resnext101_32x8d(
def resnext101_64x4d(
def wide_resnet50_2(
def wide_resnet101_2(


还可以替换输出内容。下面的命令把默认的 `num_classes: int = 1000` 修改为 `num_classes: int = 10`，并只展示相关行。

In [23]:
%%bash
sed 's/num_classes: int = 1000/num_classes: int = 10/' resnet.py | sed -n '165,175p'


class ResNet(nn.Module):
    def __init__(
        self,
        block: type[Union[BasicBlock, Bottleneck]],
        layers: list[int],
        num_classes: int = 10,
        zero_init_residual: bool = False,
        groups: int = 1,
        width_per_group: int = 64,
        replace_stride_with_dilation: Optional[list[bool]] = None,


这里的 `|` 是管道符：前半段命令先完成替换处理，后半段命令再对结果做局部显示。后续小节会详细介绍。

默认情况下，`sed` 只会把处理结果输出到终端，不会直接改动原文件。也就是说，上面的替换仅改变了显示结果，`resnet.py` 本身并未被修改。

`sed` 的价值在于可以用脚本方式批量修改文本文件，例如把 `apt` 软件源配置替换为清华镜像源。对于 Coding Agent 来说，`sed` 也是修改代码文件的一种基础工具：它可以定位固定文本，并用命令完成可复现的替换。

如果你确实要直接修改文件，可以使用下面的命令。它会覆盖原内容，执行前建议先备份。

```bash
sed -i 's/num_classes: int = 1000/num_classes: int = 10/' resnet.py
```

## 小练习

围绕 `resnet.py` 练习查看和替换输出。

依次运行：

In [25]:
%%bash
sed -n '1,5p' resnet.py
sed -n '/^def /p' resnet.py
sed 's/num_classes: int = 1000/num_classes: int = 10/' resnet.py | sed -n '165,175p'

from functools import partial
from typing import Any, Callable, Optional, Union

import torch
import torch.nn as nn
def conv3x3(in_planes: int, out_planes: int, stride: int = 1, groups: int = 1, dilation: int = 1) -> nn.Conv2d:
def conv1x1(in_planes: int, out_planes: int, stride: int = 1) -> nn.Conv2d:
def _resnet(
def resnet18(*, weights: Optional[ResNet18_Weights] = None, progress: bool = True, **kwargs: Any) -> ResNet:
def resnet34(*, weights: Optional[ResNet34_Weights] = None, progress: bool = True, **kwargs: Any) -> ResNet:
def resnet50(*, weights: Optional[ResNet50_Weights] = None, progress: bool = True, **kwargs: Any) -> ResNet:
def resnet101(*, weights: Optional[ResNet101_Weights] = None, progress: bool = True, **kwargs: Any) -> ResNet:
def resnet152(*, weights: Optional[ResNet152_Weights] = None, progress: bool = True, **kwargs: Any) -> ResNet:
def resnext50_32x4d(
def resnext101_32x8d(
def resnext101_64x4d(
def wide_resnet50_2(
def wide_resnet101_2(

class ResNet(nn.Module):


lation: Optional[list[bool]] = None,


回答：哪些命令只是筛选行？哪条命令改变了输出内容？原文件 `resnet.py` 有没有被修改？

下一节：[06_管道与重定向.ipynb](06_管道与重定向.ipynb)。